# Milestone 4

This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

Suggested Readings:
- [Hugging Face Multiple Choice Task](https://huggingface.co/docs/transformers/tasks/multiple_choice)
- [LoRA Documentation](https://huggingface.co/docs/peft/index)
- [PyTorch Softmax](https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html)

Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

Perform the following tasks on the dataset provided as part of the Kaggle competition.

In [1]:
import pandas as pd
df=pd.read_csv("../../data/raw/train.csv")

## Multiple-Choice Data Formatting

In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

Q1. Label Encoding

Convert the answer column in train.csv into numeric labels using the following mapping:

A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?

In [2]:
df_ans = df['answer'].map({'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4})
df_ans.iloc[150]

np.int64(2)

Q2. Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:

str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [3]:
df_prompt_A=df['prompt']+" [SEP] "+df['B']
len(df_prompt_A.iloc[0])

407

## Tokenization for Multiple-Choice Models

Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

Q3. Single-Row MCQ Tokenization

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?


In [4]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

options=['A', 'B', 'C', 'D', 'E']
def tokenize_row(row, max_length=128):
    prompt = [row['prompt']] * 5
    choices = [row[option] for option in options]
    tokenized_row=tokenizer(prompt, choices, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")
    return tokenized_row


row_0_tokenized=tokenize_row(df.iloc[0])['input_ids']
print(row_0_tokenized.shape)
row_0_tokenized=row_0_tokenized.unsqueeze(0)
print(row_0_tokenized.shape)
row_0_tokenized.shape[1]

torch.Size([5, 128])
torch.Size([1, 5, 128])


5

Q4. Batch MCQ Tokenization

Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [5]:
tokenized_rows=None
for i in range(16):
    row_tokenized=tokenize_row(df.iloc[i])['input_ids']
    row_tokenized=row_tokenized.unsqueeze(0)
    if tokenized_rows is None:
        tokenized_rows=row_tokenized
    else:
        tokenized_rows=torch.cat((tokenized_rows, row_tokenized), dim=0)
print(tokenized_rows.shape)
#No of Token Positions -
tokenized_rows.numel()

torch.Size([16, 5, 128])


10240

## Multiple-Choice Model Outputs

AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

Q5. Multiple-Choice Logits

Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [6]:
from transformers import AutoModelForMultipleChoice
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
inputs=tokenized_rows[:5]
outputs=model(inputs)
print("Logits shape:", outputs.logits.shape)
print("No of logits for each question:", outputs.logits.shape[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([5, 5])
No of logits for each question: 5


Q6. Supervised Loss Tensor

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [7]:
tokenized_row_0=tokenize_row(df.iloc[0])
# Add batch dimension
inputs = {
    k: v.unsqueeze(0)
    for k, v in tokenized_row_0.items()
}
inputs.keys()
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
label = torch.tensor([
    label_map[df.iloc[0]["answer"]]
])
outputs = model(**inputs, labels=label)

print("Loss:", outputs.loss.item())
print("Loss Shape:", outputs.loss.shape)
print("Logits shape:", outputs.logits.shape)

Loss: 1.608485460281372
Loss Shape: torch.Size([])
Logits shape: torch.Size([1, 5])


## LoRA for Efficient Fine-Tuning

LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

Q7. LoRA Trainable Parameters

Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [8]:
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

#Appling lora
model=get_peft_model(model, lora_config)

sum(p.numel() for p in model.parameters() if p.requires_grad)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


295681

## Preparing Data for Hugging Face Trainer
    
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

Q8. Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [9]:
from datasets import Dataset

processed_data=[]
for i, row in df[:100].iterrows():
    tokenized_row=tokenize_row(row)
    tokenized_row['labels'] = [label_map[row["answer"]]]
    processed_data.append(tokenized_row)
dataset=Dataset.from_list(processed_data)
print(dataset)
# No of tokenized choices for each question
len(dataset[0]['input_ids'])

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 100
})


5

## Tiny Fine-Tuning and Inference

In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Q9. Tiny LoRA Fine-Tuning

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [13]:
from transformers import Trainer, TrainingArguments

proccessed_data_q9=[]
for i, row in df[:32].iterrows():
    tokenized_row=tokenize_row(row, max_length=64)
    tokenized_row['labels'] = label_map[row["answer"]]
    proccessed_data_q9.append(tokenized_row)

dataset_q9=Dataset.from_list(proccessed_data_q9)

training_args = TrainingArguments(
    output_dir="./tily_lora_tuning_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
)

def data_collator(data):
    return {
        'input_ids': torch.tensor([item['input_ids'] for item in data], dtype=torch.long),
        'attention_mask': torch.tensor([item['attention_mask'] for item in data], dtype=torch.long),
        'labels': torch.tensor([item['labels'] for item in data], dtype=torch.long)
    }

trainer = Trainer(
    model=model,
    train_dataset=dataset_q9,
    args=training_args,
    data_collator=data_collator
)

train_output = trainer.train()
print("Output", train_output)
print("Final global step:", train_output.global_step)

Step,Training Loss


Output TrainOutput(global_step=4, training_loss=1.6581330299377441, metrics={'train_runtime': 2.0232, 'train_samples_per_second': 7.908, 'train_steps_per_second': 1.977, 'total_flos': 2640170250240.0, 'train_loss': 1.6581330299377441, 'epoch': 0.5})
Final global step: 4


Q10. Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [14]:
row_0=df.iloc[0]
inputs = tokenize_row(row_0, max_length=64)

model.eval()
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

# add batch dimension and move to device
inputs = {k: torch.tensor(v).unsqueeze(0).to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    # apply softmax to get probabilities
    probabilities = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

print("Probabilities:", probabilities)

# prob for option E (4th index)
print("Probability for option E:", round(probabilities[4].item(), 4))

C:\Users\dhruv\AppData\Local\Temp\ipykernel_29660\4001624351.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  inputs = {k: torch.tensor(v).unsqueeze(0).to(device) for k, v in inputs.items()}


Probabilities: [0.2003097  0.19982377 0.19939263 0.20027699 0.20019692]
Probability for option E: 0.2002
